# Dataset integrity and privacy audit

Privacy rule: no subject, body, preview, or attachment content is loaded.

In [ ]:
PROJECT_ROOT = '..'
RANDOM_SEED = 365

In [ ]:
from pathlib import Path
ROOT = Path(PROJECT_ROOT).resolve()
DATA_ROOT = ROOT / 'data'
ARTIFACT_ROOT = ROOT / 'artifacts/models/current'

In [ ]:
import hashlib, json, pandas as pd, matplotlib.pyplot as plt
manifest = json.loads((DATA_ROOT/'DATASET_MANIFEST.json').read_text())
entries = manifest['public_datasets'] + manifest['generated_datasets']
for item in entries:
    path = ROOT/item['path']; assert path.stat().st_size == item['bytes']; assert hashlib.sha256(path.read_bytes()).hexdigest() == item['sha256']
public = pd.DataFrame(manifest['public_datasets'])
display(public[['name','bytes']])
print('Declared SpamAssassin messages:', sum(item.get('messages',0) for item in manifest['public_datasets']))
print('Declared Enron files:', next(item['files'] for item in manifest['public_datasets'] if 'Enron' in item['name']))
public.plot.barh(x='name', y='bytes', legend=False, title='Compressed source size'); plt.tight_layout()

In [ ]:
daily = pd.read_csv(DATA_ROOT/'synthetic/m365_user_daily.csv.gz')
print(daily.shape, daily.isna().sum().sum())
fig, axes = plt.subplots(1,2,figsize=(11,4)); daily['label'].value_counts().plot.bar(ax=axes[0], title='Synthetic class balance'); daily.isna().mean().nlargest(10).plot.bar(ax=axes[1], title='Largest missingness rates'); plt.tight_layout()

In [ ]:
import pyarrow.dataset as ds
FORBIDDEN = {'subject','body','preview','unique_body','attachment_bytes'}
processed = ds.dataset(DATA_ROOT/'processed/email_metadata_v1', format='parquet', partitioning='hive')
assert FORBIDDEN.isdisjoint(name.lower() for name in processed.schema.names)
assert FORBIDDEN.isdisjoint(daily.columns.str.lower())
print('Privacy contract verified: forbidden content columns are absent from CSV and Parquet.')